# Staged Multi-Molecule SFT On Kaggle

This notebook resolves the grouped multi-molecule training file produced by the merged BioT5 collection stage from Kaggle artifacts or local outputs, derives grouped train/validation/test splits, runs staged multi-molecule SFT, and exports the checkpoint bundle for PPO.


In [ ]:
from pathlib import Path
import sys

REPO_URL = "https://github.com/mruniverse8/Thesis.git"
REPO_BRANCH = "gflownet"
REPO_DIR = Path("/kaggle/working/Thesis")
STAGE_NAME = "train_multi_molecule_sft"
UPSTREAM_COLLECTION_STAGE = "merge_biot5_collection_parts"

%cd /kaggle/working
!if [ -d "{REPO_DIR / '.git'}" ]; then echo "Reusing {REPO_DIR}"; elif [ -d "{REPO_DIR}" ]; then echo "Existing non-git directory at {REPO_DIR}; delete it and rerun the notebook." && false; else git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"; fi
!git -C "{REPO_DIR}" fetch --depth 1 origin "{REPO_BRANCH}" && git -C "{REPO_DIR}" checkout -B "{REPO_BRANCH}" FETCH_HEAD

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from thesis_kaggle_support import (
    copy_stage_artifact_to_local,
    ensure_grouped_split_files,
    ensure_paths_exist,
    ensure_repo_selfies_vocab,
    ensure_runtime_dependencies,
    dump_yaml,
    export_stage_artifacts,
    json_dumps,
    load_yaml,
    read_json,
    report_runtime,
)


In [ ]:
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
NUM_EPOCHS = 1
NUM_WORKERS = 2
VALIDATION_FRACTION = 0.05
TEST_FRACTION = 0.05
LOCAL_GROUPED_TRAIN_FILE = REPO_DIR / "data" / "post_training" / "processed" / "train_multimol.jsonl"
SPLIT_OUTPUT_DIR = REPO_DIR / "kaggle" / "generated_data" / "post_training_splits"
OUTPUT_DIR = REPO_DIR / "outputs" / "kaggle" / "multi_molecule_sft"
TEMP_CONFIG_PATH = REPO_DIR / "kaggle" / "generated_configs" / "multi_molecule_sft.kaggle.yaml"

ensure_runtime_dependencies(REPO_DIR)
runtime_report = report_runtime(require_gpu=True)
print(json_dumps({
    "runtime": runtime_report,
    "output_dir": str(OUTPUT_DIR),
    "split_output_dir": str(SPLIT_OUTPUT_DIR),
}))


In [ ]:
if not LOCAL_GROUPED_TRAIN_FILE.exists():
    copied_grouped_train = copy_stage_artifact_to_local(
        stage_name=UPSTREAM_COLLECTION_STAGE,
        artifact_relpath="post_training_processed/train_multimol.jsonl",
        local_path=LOCAL_GROUPED_TRAIN_FILE,
    )
else:
    copied_grouped_train = None

if not LOCAL_GROUPED_TRAIN_FILE.exists():
    raise FileNotFoundError(
        f"Grouped training file not found at {LOCAL_GROUPED_TRAIN_FILE}. Attach the collection artifact dataset or run 00_collect_chebi_biot5.ipynb first."
    )

split_paths = ensure_grouped_split_files(
    LOCAL_GROUPED_TRAIN_FILE,
    SPLIT_OUTPUT_DIR,
    seed=42,
    validation_fraction=VALIDATION_FRACTION,
    test_fraction=TEST_FRACTION,
)
print(json_dumps({
    "copied_grouped_train": None if copied_grouped_train is None else str(copied_grouped_train),
    "grouped_train_file": str(LOCAL_GROUPED_TRAIN_FILE),
    "split_paths": {name: str(path) for name, path in split_paths.items()},
}))

config = load_yaml(REPO_DIR / "configs" / "multi_molecule_sft.yaml")
config["data"]["train_file"] = str(split_paths["train"])
config["data"]["validation_file"] = str(split_paths["validation"])
config["data"]["test_file"] = str(split_paths["test"])
config["data"]["num_workers"] = int(NUM_WORKERS)
config["training"]["output_dir"] = str(OUTPUT_DIR)
config["training"]["per_device_train_batch_size"] = int(PER_DEVICE_TRAIN_BATCH_SIZE)
config["training"]["per_device_eval_batch_size"] = int(PER_DEVICE_EVAL_BATCH_SIZE)
config["training"]["gradient_accumulation_steps"] = int(GRADIENT_ACCUMULATION_STEPS)
config["training"]["num_epochs"] = int(NUM_EPOCHS)
dump_yaml(config, TEMP_CONFIG_PATH)
print(f"Wrote config: {TEMP_CONFIG_PATH}")
print(TEMP_CONFIG_PATH.read_text(encoding="utf-8"))


In [ ]:
%cd {REPO_DIR}
!python scripts/train_multi_molecule_sft.py --config "{TEMP_CONFIG_PATH}"


In [ ]:
required_outputs = ensure_paths_exist({
    "output_dir": OUTPUT_DIR,
    "best_checkpoint": OUTPUT_DIR / "checkpoints" / "best",
    "run_summary": OUTPUT_DIR / "run_summary.json",
    "history": OUTPUT_DIR / "history.json",
    "resolved_config": OUTPUT_DIR / "resolved_config.yaml",
    "tokenizer_metadata": OUTPUT_DIR / "tokenizer_metadata.json",
    "split_train": split_paths["train"],
    "split_validation": split_paths["validation"],
    "split_test": split_paths["test"],
})
artifact_dir, manifest = export_stage_artifacts(
    stage_name=STAGE_NAME,
    artifact_map={
        "grouped_splits": SPLIT_OUTPUT_DIR,
        "checkpoints": OUTPUT_DIR / "checkpoints",
        "run_summary.json": OUTPUT_DIR / "run_summary.json",
        "history.json": OUTPUT_DIR / "history.json",
        "resolved_config.yaml": OUTPUT_DIR / "resolved_config.yaml",
        "tokenizer_metadata.json": OUTPUT_DIR / "tokenizer_metadata.json",
    },
    metadata={
        "required_outputs": required_outputs,
        "config_path": str(TEMP_CONFIG_PATH),
        "summary": read_json(OUTPUT_DIR / "run_summary.json"),
    },
)
print(json_dumps({
    "artifact_dir": str(artifact_dir),
    "manifest": manifest,
}))
